In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated,Literal
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from dotenv import load_dotenv
import os
from langchain_core.messages import BaseMessage,SystemMessage, HumanMessage
from IPython.display import Image
from pydantic import BaseModel,Field
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
import operator
from langgraph.checkpoint.memory import MemorySaver # for Persistence

c:\Users\Sachidanand Sharma\Desktop\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

True

In [3]:
class ChatState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

In [5]:
def gen_joke(state:ChatState):
    prompt=f"Generate a joke on the topic {state['topic']}"
    res=llm.invoke(prompt).content
    return {'joke':res}

In [6]:
def exp_joke(state: ChatState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}


In [7]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)
graph.add_node('gen_joke',gen_joke)
graph.add_node('exp_joke',exp_joke)

graph.add_edge(START,'gen_joke')
graph.add_edge('gen_joke','exp_joke')
graph.add_edge('exp_joke',END)

workflow=graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
res=workflow.invoke({'topic':'pizza'}, config=config1)

In [9]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. \n\nIn everyday language, "feeling a little crusty" is an idiomatic expression that means being irritable, grumpy, or having a rough demeanor. However, in the context of a pizza, "crusty" has a literal meaning, referring to the crust of the pizza, which is the outer layer of the bread.\n\nThe joke relies on this double meaning of "crusty" to create humor. It sets up the expectation that the pizza is in a bad mood, and then subverts it by using the word "crusty" in a way that is both a common phrase for being grumpy and a literal description of the pizza\'s crust. This wordplay creates a clever and amusing connection between the setup and the punchline, making the joke funny and lighthearted.'}, next=(), config={'configurable': {'thread_id': '1', 'check

In [10]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. \n\nIn everyday language, "feeling a little crusty" is an idiomatic expression that means being irritable, grumpy, or having a rough demeanor. However, in the context of a pizza, "crusty" has a literal meaning, referring to the crust of the pizza, which is the outer layer of the bread.\n\nThe joke relies on this double meaning of "crusty" to create humor. It sets up the expectation that the pizza is in a bad mood, and then subverts it by using the word "crusty" in a way that is both a common phrase for being grumpy and a literal description of the pizza\'s crust. This wordplay creates a clever and amusing connection between the setup and the punchline, making the joke funny and lighthearted.'}, next=(), config={'configurable': {'thread_id': '1', 'chec

In [11]:
print(res['joke'])

Why was the pizza in a bad mood?

Because it was feeling a little crusty.


In [12]:
res['explanation']

'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. \n\nIn everyday language, "feeling a little crusty" is an idiomatic expression that means being irritable, grumpy, or having a rough demeanor. However, in the context of a pizza, "crusty" has a literal meaning, referring to the crust of the pizza, which is the outer layer of the bread.\n\nThe joke relies on this double meaning of "crusty" to create humor. It sets up the expectation that the pizza is in a bad mood, and then subverts it by using the word "crusty" in a way that is both a common phrase for being grumpy and a literal description of the pizza\'s crust. This wordplay creates a clever and amusing connection between the setup and the punchline, making the joke funny and lighthearted.'

In [19]:
config2 = {"configurable": {"thread_id": "2"}}
res2=workflow.invoke({'topic':'onion'}, config=config2)

In [20]:
res2

{'topic': 'onion',
 'joke': 'Why did the onion go to therapy?\n\nBecause it was feeling layered with emotions and was struggling to peel away its problems.',
 'explanation': 'This joke is a play on words, using the characteristics of an onion to create a pun. The joke starts by setting up a scenario where an onion goes to therapy, which is an unexpected and humorous situation. The punchline relies on the double meaning of two phrases: "feeling layered with emotions" and "peel away its problems."\n\nOnions are known for their layered structure, with each layer being a thin, papery skin that can be peeled away to reveal the next layer. In this joke, the phrase "feeling layered with emotions" is a clever play on words, as it references both the onion\'s physical structure and the idea that the onion is experiencing complex, layered emotions.\n\nThe second part of the punchline, "struggling to peel away its problems," is another clever use of wordplay. In a literal sense, peeling an onion 

### Time Travel

In [ ]:
list(workflow.get_state_history(config2))


[StateSnapshot(values={'topic': 'onion', 'joke': 'Why did the onion go to therapy?\n\nBecause it was feeling layered with emotions and was struggling to peel away its problems.', 'explanation': 'This joke is a play on words, using the characteristics of an onion to create a pun. The joke starts by setting up a scenario where an onion goes to therapy, which is an unexpected and humorous situation. The punchline relies on the double meaning of two phrases: "feeling layered with emotions" and "peel away its problems."\n\nOnions are known for their layered structure, with each layer being a thin, papery skin that can be peeled away to reveal the next layer. In this joke, the phrase "feeling layered with emotions" is a clever play on words, as it references both the onion\'s physical structure and the idea that the onion is experiencing complex, layered emotions.\n\nThe second part of the punchline, "struggling to peel away its problems," is another clever use of wordplay. In a literal sens

In [ ]:
workflow.get_state({"configurable":{"thread_id":"2","checkpoint_id":"1f168dfa-33be-6cbc-8000-564acd48b168"}})

StateSnapshot(values={'topic': 'pasta'}, next=('gen_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f168dfa-33be-6cbc-8000-564acd48b168'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-15T17:28:35.080503+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f168dfa-33bd-600d-bfff-fd9949dd30d0'}}, tasks=(PregelTask(id='7501df92-b9a7-7629-dfc8-73323cbe2391', name='gen_joke', path=('__pregel_pull', 'gen_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.'}),), interrupts=())

In [23]:
workflow.invoke(None,{"configurable":{"thread_id":"2","checkpoint_id":"1f168dfa-33be-6cbc-8000-564acd48b168"}})

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.',
 'explanation': 'This joke is a play on words, using the physical properties of spaghetti to create a pun. Spaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. \n\nIn this joke, the phrase "tangled up in a relationship" has a double meaning. In a romantic context, "tangled up" can refer to the complexities and challenges that come with being in a relationship. However, when applied to spaghetti, it also literally refers to the physical act of the pasta becoming knotted or entwined.\n\nThe humor comes from the unexpected twist on the common phrase, as the listener is initially expecting a typical reason for not wanting to get married, but instead, the joke relies on the clever wordplay to create a humorous connection between the characteristics of spaghetti and the concept of a romantic relationship. The

In [24]:
list(workflow.get_state_history(config2))


[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.', 'explanation': 'This joke is a play on words, using the physical properties of spaghetti to create a pun. Spaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. \n\nIn this joke, the phrase "tangled up in a relationship" has a double meaning. In a romantic context, "tangled up" can refer to the complexities and challenges that come with being in a relationship. However, when applied to spaghetti, it also literally refers to the physical act of the pasta becoming knotted or entwined.\n\nThe humor comes from the unexpected twist on the common phrase, as the listener is initially expecting a typical reason for not wanting to get married, but instead, the joke relies on the clever wordplay to create a humorous connection between the characteristics of spaghetti and the concept of a romant